# Cleanup and Setup

In [0]:
%run ./02-Batch-Word-Count.py

In [0]:
class batchWCTestSuit():
    def __init__(self):
        self.catalog_name='cat_spark_streaming'
        self.schema_name='wcnt'
        self.table_name='wordcountbatch'
        self.checkpoint_name='/Volumes/cat_spark_streaming/wcnt/checkpoint/checkpoint_data/'
        self.data_path='/Volumes/cat_spark_streaming/landing/datasets'
        self.test_dataset='/Volumes/cat_spark_streaming/wcnt/sample_datasets/test_datasets/'

    def cleanTests(self):
        print(f'Starting Cleanup...', end='')
        try:
            
            #spark.sql(f'DROP TABLE IF EXISTS {self.catalog_name}.{self.schema_name}.{self.table_name}')
            dbutils.fs.rm(self.checkpoint_name,True)
            dbutils.fs.rm(self.test_dataset,True)
            
        except Exception as e:
            if "PathIsNotEmptyDirectoryException" in str(e):
                dbutils.fs.mkdirs(self.checkpoint_name)
                dbutils.fs.mkdirs(self.test_dataset)
            else:
                raise e
        print(f'Complete')

    def ingestData(self,itr):
        print(f"\tStarting Ingestion...", end= '')
        dbutils.fs.cp(f'{self.data_path}/text_data_{itr}.txt',f'{self.test_dataset}') 
        print('Completed')

    def assertResult(self,expected_count):
        from pyspark.sql.functions import substr
        actual_count=spark.sql(f"SELECT sum(wordCount) FROM {self.catalog_name}.{self.schema_name}.{self.table_name} where substr(word,1,1)=='s'").collect()[0][0]
        assert expected_count == actual_count, f"Test Failed! {actual_count}!={expected_count}"

    def runTests(self):
        self.cleanTests()
        wc=batchWC()
        print("\nTesting first iteration of batch word count ...")
        self.ingestData(1)
        wc.wordCount()
        self.assertResult(25)
        print(f'First iteration of batch word count completed')

        
        print("\nTesting second iteration of batch word count ...")
        self.ingestData(2)
        wc.wordCount()
        self.assertResult(7)
        print(f'Second iteration of batch word count completed')

        print("\nTesting Third iteration of batch word count ...")
        self.ingestData(3)
        wc.wordCount()
        self.assertResult(5)
        print(f'Third iteration of batch word count completed')


In [0]:
bwcTS=batchWCTestSuit()
bwcTS.runTests()